# Семинар 2. Типы

Теории минимум. Смысл один: **mypy находит баги, которых не видно, пока
код не упадёт у пользователя.** Всё остальное — детали.

| # | Раздел |
|---|---|
| 1 | Читаем ошибки mypy |
| 2 | Protocol |
| 3 | `list[Dog]` против `Sequence[Dog]` |
| 4 | `Any` и `type: ignore` |
| 5 | Задача |

Ниже — рабочая заготовка. Запусти и забудь: она даёт функцию `mypy()`,
которая проверяет кусок кода и показывает, что скажет mypy.

In [ ]:
import pathlib
import shutil
import subprocess
import sys
import tempfile
import textwrap

WORK = pathlib.Path(tempfile.mkdtemp(prefix="seminar02-"))

# Пустой конфиг: чтобы настройки курса не подмешивались и было видно,
# что делает сам mypy.
CONFIG = WORK / "mypy.ini"
CONFIG.write_text("[mypy]\n", encoding="utf-8")


def mypy(code: str, strict: bool = True) -> None:
    """Проверить кусок кода и показать, что скажет mypy."""
    snippet = WORK / "snippet.py"
    snippet.write_text(textwrap.dedent(code).strip() + "\n", encoding="utf-8")

    command = [
        sys.executable, "-m", "mypy", "--no-incremental",
        "--config-file", str(CONFIG), "--no-error-summary",
    ]
    if strict:
        command.append("--strict")
    command.append(str(snippet))

    result = subprocess.run(command, capture_output=True, text=True)
    output = result.stdout.replace(f"{snippet}:", "строка ").strip()
    print(output or "mypy: чисто")


print("готово, песочница:", WORK)

## 1. Читаем ошибки mypy

Ошибка всегда одного вида:

```
строка 7: error: текст  [код-ошибки]
```

Код в квадратных скобках — то, что гуглится и чем глушится точечно.
Дальше — пять сообщений, которые ты увидишь чаще всего.

**Первое: вернули не то, что обещали.**

In [ ]:
mypy("""
    def average(scores: list[float]) -> float:
        if not scores:
            return None
        return sum(scores) / len(scores)
""")

Обещали `float`, в одной ветке отдали `None`. Классика: функция работает
на всех тестах, кроме пустого списка.

**Второе: забыли, что может прийти `None`.**

In [ ]:
mypy("""
    def find(names: list[str], target: str) -> str | None:
        for name in names:
            if name == target:
                return name
        return None

    def shout(names: list[str], target: str) -> str:
        return find(names, target).upper()
""")

`Student | None` — это **два** типа, и `.upper()` есть только у одного.
Лечится проверкой на `None`, а не молчанием.

**Третье: несовместимые операнды.**

In [ ]:
mypy("""
    def line(name: str, score: float) -> str:
        return name + " — " + score
""")

Здесь mypy напечатал **две** строки на один баг: вторая — следствие
первой (сложение сломалось, результат стал `Any`). Чинишь причину —
уходят обе.

**Четвёртое: передали не тот тип.**

In [ ]:
mypy("""
    def greet(name: str) -> str:
        return f"привет, {name}"

    greet(42)
""")

**Пятое: функция без аннотаций.** Появляется только в `--strict`.

In [ ]:
mypy("""
    def add(a, b):
        return a + b
""")

Без типов mypy функцию просто не смотрит — внутри может быть что угодно.
Поэтому `--strict` и требует аннотации: непроверенная функция хуже, чем
её отсутствие, потому что создаёт ощущение проверенности.

Это все пять сообщений, которые встретятся в задаче.

## 2. Protocol

Утиная типизация, которую видит проверяльщик.

`Protocol` описывает **набор методов**, а не родителя. Подходит любой
класс с такими методами — наследоваться не нужно и импортировать протокол
тоже не нужно.

In [ ]:
mypy("""
    from typing import Protocol

    class Storage(Protocol):
        def save(self, text: str) -> None: ...

    class FileStorage:            # ничего не наследует
        def save(self, text: str) -> None:
            print("в файл:", text)

    def publish(report: str, storage: Storage) -> None:
        storage.save(report)

    publish("отчёт", FileStorage())
""")

Чисто. `FileStorage` не знает о существовании `Storage`, и всё равно
подходит — совпали методы.

А теперь опечатка в имени метода:

In [ ]:
mypy("""
    from typing import Protocol

    class Storage(Protocol):
        def save(self, text: str) -> None: ...

    class ConsoleStorage:
        def write(self, text: str) -> None:    # write, не save
            print(text)

    def publish(report: str, storage: Storage) -> None:
        storage.save(report)

    publish("отчёт", ConsoleStorage())
""")

В рантайме это упало бы `AttributeError` — когда-нибудь, у кого-нибудь.
mypy говорит сразу.

**Зачем это на практике.** Функции не нужен конкретный класс — ей нужны
три метода. Объявишь протокол на эти три метода, и в тестах подсунешь
заглушку без единого наследования.

Ровно этот баг есть в задаче.

## 3. `list[Dog]` против `Sequence[Dog]`

Вопрос с мок-интервью. Начнём с того, что удивляет.

In [ ]:
mypy("""
    class Animal: ...
    class Dog(Animal): ...

    def feed_all(animals: list[Animal]) -> None:
        for animal in animals:
            print(animal)

    dogs: list[Dog] = [Dog()]
    feed_all(dogs)
""")

`Dog` — наследник `Animal`, а `list[Dog]` вместо `list[Animal]` не
принимается. mypy даже сам пишет, куда смотреть.

**Почему.** Функция получила `list[Animal]` — значит имеет право положить
туда любое животное:

```python
def feed_all(animals: list[Animal]) -> None:
    animals.append(Cat())      # законно: Cat это Animal
```

Если бы мы передали настоящий `list[Dog]`, в нём бы оказался кот. А у
вызывающего в аннотации написано `list[Dog]`, и он спокойно сделает
`dogs[1].bark()`.

Значит `list` **инвариантен**: `list[Dog]` и `list[Animal]` не
взаимозаменяемы ни в какую сторону. Не каприз, а защита.

**Решение.** Если функция только читает — попроси `Sequence`. У него нет
`append`, поэтому подменить нечего:

In [ ]:
mypy("""
    from collections.abc import Sequence

    class Animal: ...
    class Dog(Animal): ...

    def feed_all(animals: Sequence[Animal]) -> None:
        for animal in animals:
            print(animal)

    dogs: list[Dog] = [Dog()]
    feed_all(dogs)
""")

Чисто. `Sequence` **ковариантен**: `Sequence[Dog]` годится там, где ждут
`Sequence[Animal]`.

Практическое правило, которого хватает в 95% случаев:

> **Только читаешь — проси `Sequence` (или `Iterable`, `Mapping`).
> Собираешься менять — проси `list` и мирись с инвариантностью.**

Широкий тип на входе — не педантизм: он расширяет круг тех, кто может
позвать твою функцию. Этот баг тоже есть в задаче.

## 4. `Any` и `type: ignore`

`Any` — не «любой тип». Это «не проверяй здесь ничего». Всё, чего он
касается, перестаёт проверяться.

In [ ]:
CODE = """
    from typing import Any

    def parse_scores(payload: Any) -> list[float]:
        return payload["scores"]

    scores = parse_scores({"scores": ["8", "10"]})
    print(sum(scores))
"""

print("--- обычный mypy ---")
mypy(CODE, strict=False)
print()
print("--- mypy --strict ---")
mypy(CODE, strict=True)

Обычный mypy молчит. `Any` пролез в объявленный `list[float]`, и он не
возражает — хотя внутри строки, и `sum()` упадёт в рантайме.

`--strict` включает `warn_return_any` и ловит именно это: **Any вытекает
наружу из функции, которая обещала конкретный тип.**

Вот зачем в курсе strict. Он не придирчивее — он видит на две ошибки
больше в том же файле.

**Когда `Any` уместен.** На границе с внешним миром: сырой JSON, ответ
чужого API. Но привести к настоящему типу надо сразу же:

```python
def parse_scores(payload: Any) -> list[float]:
    return [float(score) for score in payload["scores"]]
```

**`# type: ignore`.** Иногда нужен — баг в стабах библиотеки, редкий
случай, который mypy не умеет. Два правила:

```python
value = weird_lib.call()      # type: ignore[no-any-return]
```

- всегда с кодом в скобках, иначе глушится всё подряд в строке;
- рядом комментарий, почему. Без него это просто «мне было лень».

Голый `# type: ignore` в задаче не принимается — проверка его ловит.

## 5. Задача

Условие — в [`task/README.md`](task/README.md).

В `task/gradebook.py` шесть багов. Модуль импортируется, `pytest` зелёный.

```
mypy  (обычный)  → 5 ошибок
mypy --strict    → 7 ошибок
```

Каждый баг — из разобранного выше: `None` вместо числа, непроверенный
`Optional`, `str + float`, инвариантность `list`, невыполненный
`Protocol`, утечка `Any`.

```bash
cd seminars/02-typing/task
uv run mypy --strict gradebook.py     # список дел
```

Проверка — `make check-typing` из корня репозитория.

Заглушить через `# type: ignore`, `cast` или `Any` не выйдет: верификатор
проверяет это отдельными пунктами.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print("песочница удалена")